<a href="https://colab.research.google.com/github/fluids/AI4GSD/blob/main/Demo_Colab/Demo3_Smartphone_Videos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AI4GSD Demo
------------------------------------------
## Smartphone videos
------------------------------------------
#### Table of contents
#### 0. Installing requried packages
#### 1. Importing required packages
#### 2. Defining working directory
#### 3. Extracting or copying your videos/photos to desired folder
#### 4. Obtaining scales from AI, drone flight data, or manual measurements
#### 5. Obtaining segmentation of anthropogenic objects
#### 6. Obtaining grain size data
#### 7. Applying quality control

##### *** Please contact Yunxiang Chen, yunxiang.chen@pnnl.gov for reporting errors

------------------------------------------
### 0. Installing requried packages

### 0a. Install requried packages

In [1]:
%pip install -q ultralytics==8.3.186 natsort pyproj PyExifTool piexif imutils shapely psutil gdown

### 0b. Connect Google drive to Colab runtime.

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


### 0c. Create AI4GSD parent direcotry and download source codes if not found.

In [3]:
import sys, subprocess
from pathlib import Path
WorkDir = Path("/content/drive/MyDrive/AI4GSD")
update_github = True

sys.path.insert(0,str(WorkDir))
#if not WorkDir.exists():subprocess.run(["git", "clone", "--progress","https://github.com/pnnl/AI4GSD.git", str(WorkDir)],check=True)
#if update_github: subprocess.run(["git", "-C", str(WorkDir), "pull", "--ff-only", "--progress"],check=True)

### 1. Importing required packages

In [4]:
import AI4GSD

### 2. Defining working directory

In [6]:
AI4GSD.pcinfo()
foldername = '20250521_YC_WalkSmartphoneRoadTest'                          # Parent folder where you save photos/videos.
PhotoDir = str(WorkDir / 'Demo_Data' / foldername)                         # AI4GSD use string path.

Machine name: 759ae431c8de
Operating system: Ubuntu 24.04.5 LTS
Architecture: x86_64
Processor: AMD EPYC 7B12
Reported CPU frequency: 2.25 GHz
PyTorch version: 2.14.1+cpu
CUDA available: False
CUDA GPU unavailable to PyTorch


### 3. Extracting or copying your videos/photos to desired folder

In [10]:
videoname = "IMG_9683.MOV"
video = Path(PhotoDir) / "videos" / videoname
video.parent.mkdir(parents=True, exist_ok=True)
url = f"https://media.githubusercontent.com/media/pnnl/AI4GSD/main/Demo_Data/20250521_YC_WalkSmartphoneRoadTest/videos/{videoname}"
if not video.exists():subprocess.run(["curl", "-fL", "--progress-bar", "-o", str(video), url], check=True)


info = AI4GSD.readEXIFVideo(video)
print(info)









{'Width': 1080, 'Height': 1920, 'Orientation': 1, 'FrameRate': 59.9585730724971, 'Duration': 36.2083333333333, 'Make': 'Apple', 'Model': 'iPhone 14 Pro', 'DateTime': datetime.datetime(2025, 5, 21, 16, 46, 35), 'GPSInfo': {'GPSLatitude': 46.3358, 'GPSLongitude': -119.277, 'GPSAltitude': 123.879, 'GPSAltitudeRef': 0}}


In [ ]:
#%% (1) Extracting or copying your videos/photos to desired folder.
AI4GSD.getPhotos(PhotoDir,nframe=500, overwrite=False)                     # Use once.

### 4. Obtaining scales from AI, drone flight data, or manual measurements

In [ ]:
exiftool = WorkDir/"AI4GSD/exes/exiftool_linux/Image-ExifTool-13.34/exiftool"
if exists(exiftool): exiftool.chmod(0o755)

PP = AI4GSD.parameters()
PP.Directory = PhotoDir
PP.OverWriteAll = True                                                     # Overwrite everything if True.
PP.OverWriteScaleFile = False
PP.OverWriteScaleRows = True
PP.ScaleSource = 'AI'                                                      # Use AI option.
PP.ModelName = 'YOLO11m.1280.20250825'                                     # Which AI to use.
PP.ComputingSource = 'local'                                               # What computing resource to use.
PP.ConfidenceThresholdUser = 0.85                                          # Minimum confidence threshold.
PP.YOLOSave = False                                                        # Need to save AI prediction to folder.
scales =  AI4GSD.metadata(PP)

### 5. Obtaining segmentation of anthropogenic objects

In [ ]:
PP = AI4GSD.parameters()
PP.Directory = PhotoDir
PP.FolderLevel = 'L1'
PP.ModelName = 'YOLO11m.640.20250906'
PP.ObjectName = 'anthro'
PP.OverWriteSegementation = True
PP.YOLOSave = False
PP.YOLOIOU = 0.45
PP.ComputingSource = 'local'
PP.ImageSizeType = 'default'
PP.SegmentationClassName =['people','road','steel']                        # What object class need to track.
AI4GSD.background(PP)

### 6. Obtaining grain size data

In [ ]:
PP = AI4GSD.parameters()
PP.Directory = PhotoDir
PP.ModelName = 'YOLO11m.1280.20250322'
PP.YOLOSave = False
PP.SaveOverlayLabel = True
PP.OverWriteAll = True
PP.PrintOnScreen = True
PP.MaximumGrainSize = 0.15
PP.OverlayFontSize = 4
PP.OverlayLegendFontSize = 4
PP.OverlayGSDTickLineWidth = 4
gsd = AI4GSD.Photo2GSD(PP)

### 7. Applying quality control

In [ ]:
gsdqc, stqc = AI4GSD.qualityControl(PhotoDir)